# 9.14 · 神经网络可视化 / Neural Network Visualization

> **课程定位 / Where this fits**
> 第 14 课，**Part 9 · 深度学习基础**（本部分最后一课）。
> Lesson 14, **Part 9 · Deep Learning Foundations** (final lesson of this part).
>
> 神经网络常被叫"黑盒"，但其实我们能**打开看看**：可视化它学到的权重、各层激活、对样本的内部表示、以及决策边界。这既能帮你**理解模型在学什么**，也是**调试**(为什么不收敛/过拟合)的有力工具。本节用前面学过的网络，演示几种最实用的可视化。
> Neural nets are called "black boxes," but we can **open them up**: visualize learned weights, layer activations, internal representations, and decision boundaries. This helps **understand what the model learns** and is a powerful **debugging** tool (why it won't converge / overfits). We demo the most practical visualizations on nets from earlier lessons.
>
> 💼 **实战/面试视角**："怎么知道模型学到了东西 / 怎么 debug 深度模型 / t-SNE 看表示" 体现你对模型的理解深度。
> 💼 **Practical/interview angle:** "how do you know the model learned / how to debug deep models / t-SNE on representations" — shows depth of understanding.

> 📐 **符号约定 / Notation**
> - 激活(activation) —— 某一层对输入的输出 / a layer's output for an input
> - 表示(representation) —— 网络中间层对样本的向量编码 / a layer's vector encoding of a sample

> 💡 **面试相关 / Interview-relevant**
> - "怎么判断网络真的学到了有用特征"（出镜率 ★★★，看表示是否可分）
> - "t-SNE/PCA 可视化高维表示"（★★★）
> - "怎么 debug 训练不收敛"（★★★★，结合 9.11）
> - "loss landscape 是什么"（★★，可补充亮点）

---

## 学习目标 / Learning Objectives
1. 可视化第一层权重，看网络学到的"模板"。
   Visualize first-layer weights as learned "templates."
2. 观察各层激活的分布，诊断死神经元/饱和。
   Inspect activation distributions to diagnose dead/saturated neurons.
3. 用 **PCA/t-SNE** 看中间层表示是否变得可分。
   Use PCA/t-SNE to see if hidden representations become separable.
4. 画出 2D 决策边界，直观理解非线性。
   Plot 2D decision boundaries to feel nonlinearity.
5. 监控训练曲线做诊断（呼应 9.11）。
   Monitor training curves for diagnosis (echoing 9.11).

## 目录 / TOC
1. [可视化第一层权重 ⭐](#1)
2. [各层激活分布：诊断死神经元 ⭐](#2)
3. [t-SNE 看学到的表示 ⭐](#3)
4. [决策边界 + 训练曲线诊断 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 可视化第一层权重 ⭐ / Visualizing First-Layer Weights

网络第一层的每个神经元，权重和输入**维度一样**。对图像来说，可以把每个神经元的权重**重排成图像的形状**显示出来——它就像这个神经元在"寻找"的**模板/图案**：输入越像这个模板，该神经元越兴奋。
Each first-layer neuron has weights of the **same dimension as the input**. For images, we can **reshape each neuron's weights into the image shape** — it's the **template/pattern** the neuron "looks for": the more the input resembles it, the more the neuron fires.

我们在 Digits(8×8) 上训一个网络，把第一层若干神经元的权重画成 8×8 图。能看到一些类似笔画/局部结构的图案——这说明网络确实学到了有意义的特征，而不是噪声。
We train a net on Digits (8×8) and render some first-layer neurons' weights as 8×8 images. You'll see stroke-like/local patterns — evidence the net learned meaningful features, not noise.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

digits = load_digits(); X = digits.data/16.0; y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
Xtr = torch.tensor(X_tr, dtype=torch.float32); ytr = torch.tensor(y_tr)

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(64,32), nn.ReLU(), nn.Linear(32,10))   # 第一层32神经元 / 32 first-layer neurons
opt = torch.optim.Adam(net.parameters(), lr=1e-3); ce = nn.CrossEntropyLoss()
for _ in range(200):
    opt.zero_grad(); ce(net(Xtr), ytr).backward(); opt.step()

W1 = net[0].weight.detach().numpy()                   # 形状 (32, 64): 32 神经元各 64 权重 / (32 neurons, 64 weights)
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(W1[i].reshape(8, 8), cmap="RdBu_r")      # 把第 i 个神经元的64权重重排成 8×8 / reshape to image
    ax.axis("off")
fig.suptitle("第一层 16 个神经元学到的权重模板(8×8): 每个神经元在'寻找'一种局部图案")
plt.tight_layout(); plt.show()
print("红/蓝表示正/负权重; 能看到笔画状的局部结构 → 网络学到了有意义的特征(非随机噪声)")


<a id="2"></a>
## 2. 各层激活分布：诊断死神经元 ⭐ / Activation Distributions: Diagnosing Dead Neurons

把一批数据喂进网络，看每一层**激活值的分布**，能诊断很多问题：
Feed a batch through the net and look at each layer's **activation distribution** to diagnose issues:
- **死神经元(dead ReLU)**：如果某 ReLU 神经元对所有输入都输出 0（永远在负区间），它就"死了"、不再学习。**大量 0 激活**是危险信号（常因学习率过大或初始化不当，见 9.9）。
  **Dead ReLU:** if a ReLU neuron outputs 0 for all inputs (always negative), it's "dead" and stops learning. **Many zero activations** is a red flag (often from too-large LR or bad init, see 9.9).
- **饱和(saturation)**：sigmoid/tanh 的激活若都挤在 ±1 两端，梯度≈0，也学不动。
  **Saturation:** if sigmoid/tanh activations cluster at ±1, gradients ≈ 0, no learning.

下面比较**正常网络**和**学习率过大导致大量死神经元**的网络的激活分布。
Below we compare activation distributions of a **healthy net** vs one with **many dead neurons from too-large LR**.


In [ ]:
def activation_stats(lr):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,10))
    o = torch.optim.SGD(m.parameters(), lr=lr)
    for _ in range(150):
        o.zero_grad(); ce(m(Xtr), ytr).backward(); o.step()
    # 取第二个 ReLU 之后的激活 / activations after the 2nd ReLU
    feats = m[:4](torch.tensor(X_te, dtype=torch.float32)).detach().numpy()
    dead_frac = (feats == 0).mean()                    # 激活恰为0的比例 / fraction of exactly-zero activations
    return feats.ravel(), dead_frac

act_ok, dead_ok = activation_stats(lr=0.1)             # 正常 / healthy
act_bad, dead_bad = activation_stats(lr=5.0)           # 学习率过大 / LR way too large
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(act_ok, bins=50, color="#4c9"); axes[0].set_title(f"正常 LR: 死神经元比例 {dead_ok:.1%}")
axes[1].hist(act_bad, bins=50, color="#c55"); axes[1].set_title(f"LR过大: 死神经元比例 {dead_bad:.1%}")
for a in axes: a.set_xlabel("激活值"); a.set_ylabel("频数")
plt.tight_layout(); plt.show()
print(f"正常网络: {dead_ok:.1%} 激活为0 (健康, ReLU 本就该有部分为0)")
print(f"LR过大:   {dead_bad:.1%} 激活为0 (大量死神经元 → 学不动); 解决: 降LR/换He初始化/用LeakyReLU")


<a id="3"></a>
## 3. t-SNE 看学到的表示 ⭐ / t-SNE on Learned Representations

网络中间层把每个样本编码成一个**高维向量(表示)**。一个学得好的网络，会让**同类样本的表示聚到一起、不同类分开**。我们用降维（**PCA / t-SNE**，见 Part 6）把这些高维表示压到 2D 画出来，直观检验。
A hidden layer encodes each sample as a **high-dim vector (representation)**. A well-trained net makes **same-class representations cluster together, different classes separate**. We use dimensionality reduction (**PCA / t-SNE**, see Part 6) to squash these to 2D and check visually.

对比**原始像素**和**网络最后一层前的表示**：后者应该明显更"类内紧、类间分"。这是判断"网络是否真学到了有用特征"的直接证据（面试加分点）。
Comparing **raw pixels** vs **the representation before the final layer**: the latter should be clearly more "tight within class, separated between classes." Direct evidence the net learned useful features (interview bonus).


In [ ]:
from sklearn.manifold import TSNE

Xte_t = torch.tensor(X_te, dtype=torch.float32)
# 取最后一层之前的表示(倒数第二层输出) / representation before the final linear layer
with torch.no_grad():
    rep = net[:2](Xte_t).numpy()                       # net=[Linear,ReLU,Linear]; net[:2] 到 ReLU 输出
# t-SNE 把高维压到 2D / t-SNE to 2D (perplexity 适中)
ts_raw = TSNE(n_components=2, init="pca", random_state=0, perplexity=30).fit_transform(X_te)
ts_rep = TSNE(n_components=2, init="pca", random_state=0, perplexity=30).fit_transform(rep)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, title in [(axes[0], ts_raw, "原始像素 (64维)"), (axes[1], ts_rep, "网络学到的表示")]:
    sc = ax.scatter(data[:,0], data[:,1], c=y_te, cmap="tab10", s=12, alpha=0.8)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axes, label="数字类别"); fig.suptitle("t-SNE: 网络的表示比原始像素更'类内紧、类间分'")
plt.show()
print("右图比左图聚类更清晰 → 网络确实学到了把不同数字分开的有用表示")
print("用途: 验证模型学到东西 / 发现哪些类容易混淆 / 检查特征质量")


<a id="4"></a>
## 4. 决策边界 + 训练曲线诊断 + 小结 ⭐ / Decision Boundary, Curve Diagnosis & Summary

**决策边界**：在 2D 玩具数据上，可以把网络划分空间的方式整张画出来，直观看到神经网络如何用非线性"包住"复杂形状（线性模型做不到）。
**Decision boundary:** on 2D toy data, we can plot how the net partitions space, directly seeing how a neural net wraps complex shapes nonlinearly (linear models can't).


In [ ]:
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=400, noise=0.2, random_state=0)   # 两个月牙形(非线性可分) / nonlinear toy
Xm_t = torch.tensor(Xm, dtype=torch.float32); ym_t = torch.tensor(ym)
torch.manual_seed(0)
clf = nn.Sequential(nn.Linear(2,32), nn.ReLU(), nn.Linear(32,32), nn.ReLU(), nn.Linear(32,2))
o = torch.optim.Adam(clf.parameters(), lr=1e-2)
for _ in range(300): o.zero_grad(); ce(clf(Xm_t), ym_t).backward(); o.step()
# 在网格上预测, 画决策区域 / predict over a grid for the decision surface
xx, yy = np.meshgrid(np.linspace(-2,3,200), np.linspace(-1.5,2,200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad(): zz = clf(grid).argmax(1).numpy().reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7,5))
ax.contourf(xx, yy, zz, alpha=0.3, cmap="coolwarm")             # 决策区域 / decision regions
ax.scatter(Xm[:,0], Xm[:,1], c=ym, cmap="coolwarm", edgecolor="k", s=20)
ax.set_title("神经网络的决策边界: 用非线性'包住'两个月牙(线性模型做不到)")
plt.tight_layout(); plt.show()
print("弯曲的边界 = 多层+非线性激活的威力; 线性模型只能画直线, 分不开月牙")


**训练曲线诊断**（呼应 9.10/9.11）——最常用、最该养成的习惯：每次训练都画 train/val 曲线，一眼看出问题。
**Training-curve diagnosis** (echoing 9.10/9.11) — the most common habit to build: always plot train/val curves to spot issues at a glance.
- 两条都高、不降 → **欠拟合/没学动**（模型太小、LR 不对、有 bug）。
  Both high, not dropping → **underfitting** (model too small, wrong LR, a bug).
- train 低、val 高且分开 → **过拟合**（加正则/数据/早停）。
  Train low, val high and diverging → **overfitting** (add regularization/data/early stopping).
- loss 突然变 NaN/飙升 → **梯度爆炸**（降 LR / 梯度裁剪）。
  Loss suddenly NaN/spikes → **exploding gradients** (lower LR / gradient clipping).

```
第一层权重: 重排成图像看"模板", 验证学到有意义特征(非噪声)
激活分布: 大量0→死神经元(LR过大/初始化差→降LR/He/LeakyReLU); ±1饱和→梯度消失
t-SNE/PCA 表示: 好网络让同类聚、异类分; 验证特征质量/发现易混类
决策边界: 多层+非线性能包住复杂形状(线性模型只能直线)
训练曲线诊断: 都高=欠拟合; train低val高=过拟合; NaN/飙升=梯度爆炸
```

### 💡 面试速查 / Interview cheat-sheet
1. **第一层权重**: 可视化成模板, 看是否学到有意义图案。
   First-layer weights: visualize as templates, check for meaningful patterns.
2. **死神经元**: 大量 0 激活 = ReLU 死亡(降LR/He/LeakyReLU)。
   Dead neurons: many zero activations = dead ReLU (lower LR/He/LeakyReLU).
3. **t-SNE 表示**: 好网络的中间表示同类聚、异类分。
   t-SNE: a good net's hidden representations cluster by class.
4. **决策边界**: 体现多层非线性的表达力。
   Decision boundary: shows the expressive power of nonlinearity.
5. **训练曲线**: 诊断欠拟合/过拟合/梯度爆炸的第一工具。
   Training curves: the first tool to diagnose under/overfitting/explosion.

### 🎉 Part 9 完成 / Part 9 Complete
你已走完**深度学习基础**：从感知机、反向传播、PyTorch/Keras，到激活/损失/优化器/调度/初始化/正则化/训练技巧/分布式/迁移学习/可视化。这些是后续 **Part 10 计算机视觉、Part 11 NLP、Part 12 大模型** 的共同地基。
You've completed **Deep Learning Foundations**: from perceptron, backprop, PyTorch/Keras, through activations/losses/optimizers/schedulers/init/regularization/training tricks/distributed/transfer learning/visualization. These are the shared groundwork for **Part 10 Computer Vision, Part 11 NLP, Part 12 Large Models**.
